In [ ]:
"""
Asymmetric distillation factory search (Section IV of the paper).

Implements Theorem 8's decoupling condition for two-group asymmetric
borrowed identities:

    Phi(i_O, i_S) - Phi(i_O, 0) ≡ 0 (mod 2π)  for all (i_O, i_S),

where

    Phi(i_O, i_S) ≡ -θ Σ σ_{w_O, w_S} [y^w_O z^w_S] f_{i_O, i_S}(y, z)
    f_{i_O, i_S} = (1-y)^i_O (1+y)^(k-i_O) (1-z)^i_S (1+z)^(n-k-i_S)

and θ = π/2^l.

For each (l, k) the search finds the factory minimizing
γ = log_d(N/k), classified by output magic state, and compares
against Bravyi-Haah, Reed-Muller / punctured RM, and Nezami-Haah
references.
"""

from math import comb, log
from itertools import product
import time


# --- Phase polynomial ---------------------------------------------------------

def f_coefficient(i_O, i_S, k, n_minus_k, w_O, w_S):
    """
    Compute [y^w_O z^w_S] f_{i_O, i_S}(y, z) where
        f = (1-y)^i_O (1+y)^(k-i_O) (1-z)^i_S (1+z)^(n-k-i_S).

    Uses the binomial expansion: [y^w] (1-y)^a (1+y)^b
        = sum_{j} (-1)^j C(a, j) C(b, w-j).
    """
    coeff_y = sum(
        ((-1) ** j) * comb(i_O, j) * comb(k - i_O, w_O - j)
        for j in range(max(0, w_O - (k - i_O)), min(i_O, w_O) + 1)
    )
    coeff_z = sum(
        ((-1) ** j) * comb(i_S, j) * comb(n_minus_k - i_S, w_S - j)
        for j in range(max(0, w_S - (n_minus_k - i_S)), min(i_S, w_S) + 1)
    )
    return coeff_y * coeff_z


def phi_over_theta_int(i_O, i_S, k, n_minus_k, W, sigma_dict):
    """
    Compute Phi(i_O, i_S) / θ as an integer, where Phi is given by
    Lemma 7. The negative sign in the lemma is absorbed.

    Returns: integer M such that Phi(i_O, i_S) ≡ M * θ (mod 2π).
    """
    total = 0
    for (w_O, w_S) in W:
        sigma = sigma_dict.get((w_O, w_S), 0)
        if sigma == 0:
            continue
        c = f_coefficient(i_O, i_S, k, n_minus_k, w_O, w_S)
        total -= sigma * c
    return total


# --- Decoupling condition -----------------------------------------------------

def is_asymmetric_valid(l, n, k, W, sigma_dict):
    """
    Check Theorem 8: Phi(i_O, i_S) - Phi(i_O, 0) ≡ 0 (mod 2π) for all
    (i_O, i_S).

    In integer terms: Phi(i_O, i_S) ≡ Phi(i_O, 0) (mod 2^(l+1)) for
    all i_O ∈ {0, ..., k} and i_S ∈ {0, ..., n-k}.
    """
    n_minus_k = n - k
    full_mod = 1 << (l + 1)  # 2^(l+1) ≡ 2π / θ
    for i_O in range(k + 1):
        phi_at_0 = phi_over_theta_int(i_O, 0, k, n_minus_k, W, sigma_dict) % full_mod
        for i_S in range(1, n_minus_k + 1):
            phi_at_S = phi_over_theta_int(i_O, i_S, k, n_minus_k, W, sigma_dict) % full_mod
            if phi_at_S != phi_at_0:
                return False
    return True


# --- Output magic state classification ---------------------------------------

def classify_output_state(l, n, k, W, sigma_dict):
    """
    Classify the output magic state from the i_O-dependence of Phi(i_O, 0).

    - Linear in i_O: |T⟩^⊗k (unentangled T states).
    - Quadratic in i_O: contains C(i_O, 2) term, |CS⟩-type entangled state.
    - Cubic in i_O: contains C(i_O, 3) term, |CCZ⟩-type entangled state.
    - Higher: more general C_3 magic state.
    - Constant: trivial (no magic state extracted).
    """
    n_minus_k = n - k
    full_mod = 1 << (l + 1)
    # Phi(i_O, 0) values for i_O = 0, 1, ..., k
    phi_vals = [
        phi_over_theta_int(i_O, 0, k, n_minus_k, W, sigma_dict) % full_mod
        for i_O in range(k + 1)
    ]
    # Take successive differences (over the integers, before mod) to find
    # the polynomial degree in i_O. Use shifted values.
    diffs = phi_vals[:]
    degree = 0
    for d in range(1, k + 2):
        new_diffs = [diffs[j + 1] - diffs[j] for j in range(len(diffs) - 1)]
        # Reduce each entry to the canonical residue closest to 0 mod full_mod
        new_diffs = [
            ((x + full_mod // 2) % full_mod) - full_mod // 2
            for x in new_diffs
        ]
        # If all zero, polynomial degree is d-1
        if all(x == 0 for x in new_diffs):
            degree = d - 1
            break
        diffs = new_diffs
    else:
        degree = k + 1  # Higher than any polynomial of degree k

    if degree == 0:
        return "trivial"
    elif degree == 1:
        return "T"
    elif degree == 2:
        return "CS"
    elif degree == 3:
        return "CCZ"
    else:
        return f"C3-deg{degree}"


# --- Factory parameters -------------------------------------------------------

def factory_params(l, n, k, W, sigma_dict):
    """
    Compute (N, d) for a valid factory.

    N = sum_{(w_O, w_S) ∈ active} |sigma| * C(k, w_O) * C(n-k, w_S)
    d = min w_S over active gate types + 1 (since w_S ≥ d-1 ⇒ d ≤ w_S+1)
    """
    active = [(wO, wS) for (wO, wS) in W if sigma_dict.get((wO, wS), 0) != 0]
    if not active:
        return None, None
    N = sum(comb(k, wO) * comb(n - k, wS) for (wO, wS) in active)
    d = min(wS for (wO, wS) in active) + 1
    return N, d


def gamma(N, k, d):
    """γ = log_d(N/k)."""
    if k == 0 or N == 0 or d < 2:
        return float('inf')
    return log(N / k) / log(d)


# --- Search engines ----------------------------------------------------------

def gate_types(n, k, d_min):
    """All (w_O, w_S) gate types with w_S ≥ d_min - 1, w_O ≥ 0."""
    return [(w_O, w_S)
            for w_O in range(k + 1)
            for w_S in range(d_min - 1, n - k + 1)
            if w_O + w_S >= 1]  # exclude identity gate


def brute_force_search(l, n, k, d_min, max_W_size=20):
    """
    Brute-force enumerate sigma ∈ {-1, 0, +1}^|W| over all gate types
    with w_S ≥ d_min - 1. Yields valid factories.
    """
    W = gate_types(n, k, d_min)
    if len(W) > max_W_size:
        return  # too large for brute force
    for sigma_tuple in product([-1, 0, 1], repeat=len(W)):
        sigma_dict = {W[i]: sigma_tuple[i] for i in range(len(W))}
        # Skip all-zero
        if all(s == 0 for s in sigma_tuple):
            continue
        if is_asymmetric_valid(l, n, k, W, sigma_dict):
            N, d = factory_params(l, n, k, W, sigma_dict)
            if N is None:
                continue
            output = classify_output_state(l, n, k, W, sigma_dict)
            yield {
                "l": l, "n": n, "k": k, "N": N, "d": d,
                "gamma": gamma(N, k, d),
                "output": output,
                "W": W, "sigma": dict(sigma_dict),
            }


# --- Frontier extraction -----------------------------------------------------

def gamma_frontier(results):
    """
    For each (l, k, output_state) group, return the factory with smallest
    gamma. Ties broken by smallest N, then smallest n.
    """
    best = {}
    for r in results:
        key = (r["l"], r["k"], r["output"])
        if key not in best or (
            r["gamma"], r["N"], r["n"]
        ) < (
            best[key]["gamma"], best[key]["N"], best[key]["n"]
        ):
            best[key] = r
    return best


# --- Reference families ------------------------------------------------------

def bravyi_haah_gamma(k):
    """Bravyi-Haah [[3k+8, k, 2]] family, even k only."""
    if k < 2 or k % 2 != 0:
        return None
    N = 3 * k + 8
    d = 2
    return {
        "N": N, "k": k, "d": d, "gamma": gamma(N, k, d),
        "output": "T", "family": "Bravyi-Haah",
    }


def reed_muller_gamma(l):
    """Standard Reed-Muller [[2^(l+1)-1, 1, 3]] family."""
    N = (1 << (l + 1)) - 1
    d = 3
    return {
        "N": N, "k": 1, "d": d, "gamma": gamma(N, 1, d),
        "output": "T", "family": "Reed-Muller",
    }


def punctured_rm_gamma(m, r, w):
    """
    Hastings-Haah punctured Reed-Muller family
    [[C(m,>w), C(m,≤w), C(r+1,>w)]].
    """
    n_code = sum(comb(m, j) for j in range(w + 1, m + 1))
    k_code = sum(comb(m, j) for j in range(w + 1))
    d_code = sum(comb(r + 1, j) for j in range(w + 1, r + 2))
    if d_code < 2 or n_code <= k_code:
        return None
    return {
        "N": n_code, "k": k_code, "d": d_code,
        "gamma": gamma(n_code, k_code, d_code),
        "output": "T", "family": "punctured RM",
        "params": (m, r, w),
    }


# Nezami-Haah frontier points, read from their published figure.
# To be filled in by reading off Fig. X of arXiv:2107.09684.
# Format: list of (k, gamma_best, output_state, n_best) tuples.
NEZAMI_HAAH_FRONTIER = [
    # (k, gamma, output, n) — placeholder values, fill in from paper
    # Example entries, replace with actual data:
    # (1, 2.46, "T", 15),
    # (2, 1.81, "T", 14),
    # (3, 1.59, "T", 17),
    # (4, 1.32, "T", 20),
    # ...
]


# --- Comparison logic --------------------------------------------------------

def compare_to_reference(our_factory, ref_factory, tol=1e-6):
    """
    Classify our_factory vs. ref_factory: 'better', 'same', 'worse',
    or 'incomparable' if output states differ.
    """
    if our_factory["output"] != ref_factory["output"]:
        return "incomparable"
    diff = our_factory["gamma"] - ref_factory["gamma"]
    if abs(diff) < tol:
        return "same"
    elif diff < 0:
        return "better"
    else:
        return "worse"


def comparison_table(frontier):
    """
    For each (l, k, output) in frontier, compare against Bravyi-Haah,
    Reed-Muller, Nezami-Haah at matching (k, output).
    """
    rows = []
    for (l, k, output), our in frontier.items():
        row = {
            "l": l, "k": k, "output": output,
            "gamma": our["gamma"], "N": our["N"], "d": our["d"],
            "n": our["n"],
        }
        # Bravyi-Haah comparison
        bh = bravyi_haah_gamma(k)
        row["vs_BH"] = compare_to_reference(our, bh) if bh else "N/A (odd k)"
        # Reed-Muller comparison (k=1 only)
        if k == 1:
            rm = reed_muller_gamma(l)
            row["vs_RM"] = compare_to_reference(our, rm)
        else:
            row["vs_RM"] = "N/A (k>1)"
        # Nezami-Haah comparison
        nh_match = next(
            (
                {"gamma": gam, "output": out, "k": k_n, "N": None, "d": None}
                for (k_n, gam, out, _) in NEZAMI_HAAH_FRONTIER
                if k_n == k and out == output
            ),
            None,
        )
        if nh_match:
            nh_match["gamma"] = nh_match["gamma"]
            row["vs_NH"] = compare_to_reference(our, nh_match)
        else:
            row["vs_NH"] = "not in their data"
        rows.append(row)
    return rows


# --- Main search loop --------------------------------------------------------

def search_all(L_RANGE, K_RANGE, N_MAX, D_MIN_RANGE):
    """
    Run brute-force search over (l, k, n, d_min) and return all valid
    factories.
    """
    all_results = []
    timing = {}
    for l in L_RANGE:
        for k in K_RANGE:
            for n in range(k + 1, N_MAX + 1):
                if n - k < 1:
                    continue
                for d_min in D_MIN_RANGE:
                    W = gate_types(n, k, d_min)
                    if len(W) > 20:
                        continue  # skip; needs SNF/SAT
                    t_start = time.time()
                    found = list(brute_force_search(l, n, k, d_min))
                    t_elapsed = time.time() - t_start
                    timing[(l, k, n, d_min)] = {
                        "time": t_elapsed,
                        "W_size": len(W),
                        "found": len(found),
                    }
                    all_results.extend(found)
    return all_results, timing


# --- Output ------------------------------------------------------------------

def print_frontier(frontier):
    print(f"\n{'l':<3}{'k':<4}{'output':<10}{'γ':<7}{'[[N,k,d]]':<14}{'n':<4}")
    print("-" * 50)
    for (l, k, output), r in sorted(frontier.items()):
        code = f"[[{r['N']},{r['k']},{r['d']}]]"
        print(f"{l:<3}{k:<4}{output:<10}{r['gamma']:<7.3f}"
              f"{code:<14}{r['n']:<4}")


def print_comparison(comparison_rows):
    print(f"\n{'l':<3}{'k':<4}{'out':<8}{'γ':<7}{'vs BH':<14}{'vs RM':<14}{'vs NH':<18}")
    print("-" * 70)
    for r in sorted(comparison_rows, key=lambda x: (x["l"], x["k"], x["output"])):
        print(f"{r['l']:<3}{r['k']:<4}{r['output']:<8}{r['gamma']:<7.3f}"
              f"{r['vs_BH']:<14}{r['vs_RM']:<14}{r['vs_NH']:<18}")


def print_timing(timing):
    print("\nSearch-space and timing:")
    print(f"{'l':<3}{'k':<4}{'n':<4}{'d_min':<6}{'|W|':<5}{'#found':<7}{'time(s)':<10}")
    for (l, k, n, d_min), info in sorted(timing.items()):
        print(f"{l:<3}{k:<4}{n:<4}{d_min:<6}{info['W_size']:<5}"
              f"{info['found']:<7}{info['time']:<10.3f}")


# --- Main --------------------------------------------------------------------

if __name__ == "__main__":
    L_RANGE = [2, 3, 4]
    K_RANGE = [1, 2, 3, 4, 5, 6]
    N_MAX = 9
    D_MIN_RANGE = [2, 3, 4]

    print("=" * 60)
    print("Asymmetric distillation factory search (Section IV)")
    print(f"l ∈ {L_RANGE}, k ∈ {K_RANGE}, n ≤ {N_MAX}, d_min ∈ {D_MIN_RANGE}")
    print("=" * 60)

    all_results, timing = search_all(L_RANGE, K_RANGE, N_MAX, D_MIN_RANGE)
    print(f"\nTotal valid factories found: {len(all_results)}")

    frontier = gamma_frontier(all_results)
    print_frontier(frontier)

    rows = comparison_table(frontier)
    print_comparison(rows)

    print_timing(timing)

    # Sanity checks
    print("\n=== Sanity checks ===")
    bh4 = bravyi_haah_gamma(4)
    print(f"Bravyi-Haah at k=4: {bh4}")
    bh4_in_frontier = frontier.get((3, 4, "T"))
    if bh4_in_frontier:
        print(f"Our (l=3, k=4, T): γ={bh4_in_frontier['gamma']:.3f}, "
              f"[[{bh4_in_frontier['N']},{bh4_in_frontier['k']},{bh4_in_frontier['d']}]]")
        print(f"Should be [[20, 4, 2]]: γ = {gamma(20, 4, 2):.3f}")